# 🧠 StudySync — Ollama AI Backend

This notebook sets up a local Ollama server with **Mistral** and exposes it publicly via **ngrok** so StudySync can use it as the AI backend.

### ✅ Steps
1. Install Ollama
2. Pull the Mistral model
3. Start Ollama server
4. Expose via ngrok
5. Copy the public URL → paste into StudySync backend `.env`

> **Reminder:** Go to `Runtime > Change runtime type` and select **GPU (T4)** before running.

## 📦 Step 1 — Install Ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

## 🤖 Step 2 — Start Ollama Server (runs in background)

In [ ]:
import subprocess
import time

# Start Ollama server in the background
ollama_process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Give it a moment to spin up
time.sleep(3)
print('✅ Ollama server started!')

## 📥 Step 3 — Pull the Model

Change `mistral` to `phi3` if you're on a CPU-only runtime.

In [ ]:
# Change to 'phi3' if no GPU available
MODEL = 'mistral'

!ollama pull {MODEL}
print(f'✅ Model {MODEL} ready!')

## 🔌 Step 4 — Install & Configure ngrok

Get your free auth token from https://dashboard.ngrok.com/authtokens and paste it below.

In [ ]:
!pip install pyngrok -q

from pyngrok import ngrok

# 🔑 Paste your ngrok auth token here
NGROK_AUTH_TOKEN = 'YOUR_NGROK_TOKEN_HERE'

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print('✅ ngrok configured!')

## 🌐 Step 5 — Expose Ollama Publicly

In [ ]:
# Expose Ollama's default port
public_url = ngrok.connect(11434)

print('='*50)
print('🚀 Ollama is publicly accessible!')
print(f'\n📋 Your public URL:\n\n   {public_url}\n')
print('👉 Copy this URL and set it in your StudySync backend .env as:')
print(f'   OLLAMA_BASE_URL={public_url}')
print('='*50)

## 🧪 Step 6 — Test It (Optional)

Send a quick message to confirm the model is responding.

In [ ]:
import requests
import json

test_response = requests.post(
    'http://localhost:11434/api/chat',
    json={
        'model': MODEL,
        'messages': [{'role': 'user', 'content': 'Give me one quick study tip.'}],
        'stream': False
    }
)

result = test_response.json()
print('🤖 Model response:')
print(result['message']['content'])

## ⏹️ Stop Server (Run when done)

In [ ]:
ngrok.kill()
ollama_process.terminate()
print('🛑 Ollama server and ngrok tunnel stopped.')

---
## 📝 StudySync Backend Integration

In your `backend/services/services.js`, replace the Gemini AI call with:

```javascript
const axios = require('axios');

async function aiChat(userMessage) {
  const response = await axios.post(`${process.env.OLLAMA_BASE_URL}/api/chat`, {
    model: process.env.OLLAMA_MODEL || 'mistral',
    messages: [{ role: 'user', content: userMessage }],
    stream: false
  });
  return response.data.message.content;
}

module.exports = { aiChat };
```

And in your `.env`:
```
OLLAMA_BASE_URL=https://your-ngrok-url-here
OLLAMA_MODEL=mistral
```